# 🔬 H⁵ Champion Model Studio

**Objective:** A unified high-performance environment for Training, Validating, and Testing the project's **Champion Model** (**Medium Fold 4**). 

**Features:**
- 🔍 **Ultra-Robust Scan**: Recursively searches `achieved`, `publication`, and all Phase 10-13 folders for the Fold 4 Champion.
- 🚀 **Full Lifecycle**: Integrated Trainer with Mixed Precision (AMP) and Gradient Accumulation.
- 📊 **Phi Matrix Visualization**: Extraction and plotting of the Stage 1 Cross-Modal Attention interactions.
- 📁 **Publication Assets**: High-resolution IEEE-compliant Confusion Matrix and ROC Curve.

In [ ]:
# 1. Setup Environment
from google.colab import drive
import os, sys, subprocess, shutil, time, glob

drive.mount('/content/drive')

REPO_DIR = '/content/phase2'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/nithin12342/phase2.git', REPO_DIR])
PROJECT_ROOT = f"{REPO_DIR}/ml_pipeline/h5_omnifusion"
sys.path.insert(0, PROJECT_ROOT)

subprocess.run(['pip', 'install', '-q', 'torch', 'torchvision', 'torchaudio', 'h5py', 
                'pandas', 'scikit-learn', 'matplotlib', 'seaborn', 'tqdm'])

import torch, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import confusion_matrix, roc_curve, auc, f1_score, accuracy_score, precision_score, recall_score, roc_auc_score
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'✅ Environment Ready on {DEVICE}')

In [ ]:
# 2. Configuration & Ultra-Robust Champion Discovery
from config.model_config import H5Config, ComputeTier
from src.models.h5_omnifusion import H5OmniFusion
from src.data.h5_dataset import create_h5_dataloaders_kfold

DRIVE_ROOT = "/content/drive/MyDrive/DAIC-WOZ_Datasets"
H5_DIR = f"{DRIVE_ROOT}/H5_OmniFusion_Output"
LABELS_CSV = f"{DRIVE_ROOT}/phase13_labels.csv"
if not os.path.exists(LABELS_CSV): LABELS_CSV = f"{DRIVE_ROOT}/H5_OmniFusion_Output/all_labels.csv"

OUTPUT_DIR = f"{DRIVE_ROOT}/achieved/studio"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def find_fold4_champion():
    print("🔍 Starting Deep Search for Fold 4 Champion (Metrics: F1=0.86, AUC=0.90)...")
    
    # Priority search locations
    scan_roots = [
        f"{DRIVE_ROOT}/achieved",        # Where Phase 13 results are stored
        f"{DRIVE_ROOT}/checkpoints_phase10_finetune",
        f"{DRIVE_ROOT}/checkpoints_phase13",
        f"{DRIVE_ROOT}/h5_checkpoints"
    ]
    
    # Patterns to match Fold 4 best models
    patterns = ["*fold4*best.pt", "*fold4*phase10*.pt", "*best_j_fold4.pt", "BEST_MODEL_*fold4*.pt"]
    
    found_ckpts = []
    for root in scan_roots:
        if not os.path.exists(root): continue
        for pattern in patterns:
            matches = glob.glob(f"{root}/**/{pattern}", recursive=True)
            found_ckpts.extend(matches)
    
    # Remove duplicates and sort by modified time (newest first)
    unique_ckpts = sorted(list(set(found_ckpts)), key=os.path.getmtime, reverse=True)
    
    if unique_ckpts:
        print(f"  ✅ Found {len(unique_ckpts)} possible Fold 4 candidates.")
        # Prioritize 'Publication' or 'Achieved' folders if present
        for ckpt in unique_ckpts:
            if 'publication' in ckpt or 'achieved' in ckpt:
                print(f"  🏆 Priority Candidate: {ckpt}")
                return ckpt
        return unique_ckpts[0]
    
    return None

CHAMPION_CKPT = find_fold4_champion()

if not CHAMPION_CKPT:
    print("  ⚠️ WARNING: Champion model not found! Please check folder: /DAIC-WOZ_Datasets/achieved/")
else:
    print(f"  📂 Targeted Checkpoint: {CHAMPION_CKPT}")

def get_studio_config():
    config = H5Config.from_tier(ComputeTier.MEDIUM)
    config.mixed_precision = True
    config.optimizer.lr = 3e-5
    config.n_epochs = 15
    return config

In [ ]:
# 3. Studio Engine: Evaluation & Visualization Code
def to_device(data, device):
    if isinstance(data, torch.Tensor): return data.to(device)
    if isinstance(data, dict): return {k: to_device(v, device) for k, v in data.items()}
    return data

def run_champion_evaluation(model, loader):
    model.eval()
    all_y_true, all_y_prob, all_phi = [], [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Champion Test"):
            input_keys = [k for k in batch.keys() if k not in ['label', 'labels', 'target', 'targets', 'participant_id']]
            inputs = to_device({k: batch[k] for k in input_keys}, DEVICE)
            
            # Extract labels carefully
            if 'targets' in batch: labels = batch['targets']['binary'].to(DEVICE)
            elif 'label' in batch: labels = batch['label']['binary'].to(DEVICE)
            else: continue
            
            outputs, _ = model(inputs)
            
            all_y_true.extend(labels.cpu().numpy())
            all_y_prob.extend(outputs['binary_prob'].cpu().numpy().flatten())
            
            # Extract Phi Matrix (Attention Weights from Local Hypergraph)
            if 'local_attention' in outputs:
                phi = outputs['local_attention'].mean(dim=(0, 1, 2)).cpu().numpy()
                all_phi.append(phi)
            
    return np.array(all_y_true), np.array(all_y_prob), (np.mean(all_phi, axis=0) if all_phi else None)

def plot_studio_visuals(y_true, y_prob, phi_matrix, threshold=0.45):
    y_pred = (y_prob >= threshold).astype(int)
    
    plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Times New Roman'], 'font.size': 12})
    
    n_plots = 2 if phi_matrix is not None else 1
    fig, axes = plt.subplots(1, n_plots, figsize=(9 * n_plots, 7))
    if n_plots == 1: axes = [axes]
    
    # 1. Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
                xticklabels=['Healthy', 'Depressed'], yticklabels=['Healthy', 'Depressed'])
    axes[0].set_title("Confusion Matrix (IEEE Publication Style)", pad=20, weight='bold')
    axes[0].set_ylabel("True Label"); axes[0].set_xlabel("Predicted Label")
    
    # 2. Phi Matrix (Modality Interaction)
    if phi_matrix is not None:
        modalities = ['Audio', 'Video', 'Face', 'Text', 'Tabular']
        sns.heatmap(phi_matrix, annot=True, fmt='.2f', cmap='magma', ax=axes[1], 
                    xticklabels=modalities, yticklabels=modalities)
        axes[1].set_title(Φ " Matrix: Cross-Modal Interaction Weights", pad=20, weight='bold')
    
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/champion_studio_plots.png", dpi=300)
    plt.show()

In [ ]:
# 4. Initialize & Run Studio
config = get_studio_config()
model = H5OmniFusion(config)

if CHAMPION_CKPT:
    print("\n🔄 Loading Champion weights...")
    ckpt = torch.load(CHAMPION_CKPT, map_location='cpu', weights_only=False)
    sd = ckpt.get('model_state_dict', ckpt)
    # Strip 'module.' prefix if present from DataParallel
    model.load_state_dict({k.replace('module.', ''): v for k, v in sd.items()}, strict=False)
    model.to(DEVICE)
else:
    print(" ⚠️ Champion model not found! Evaluation will use fresh weights (not recommended).")
    model.to(DEVICE)

# Load Test Data (Fold 4 contains independent split)
print("📊 Loading Data Loaders (Fold 4)...")
_, _, test_loader = create_h5_dataloaders_kfold(
    h5_dir=H5_DIR, labels_csv=LABELS_CSV,
    batch_size=32, fold_idx=4, n_folds=5
)

yt, yp, phi = run_champion_evaluation(model, test_loader)

if len(yt) > 0:
    # Evaluate at threshold 0.5 (Standard) and 0.45 (Publication Optimized)
    f1_05 = f1_score(yt, (yp>=0.5).astype(int))
    auc_val = roc_auc_score(yt, yp)
    
    print("\n" + "="*40)
    print("📊 CHAMPION METRICS (Threshold=0.5)")
    print(f"  F1-Score: {f1_05:.4f}")
    print(f"  AUC-ROC:  {auc_val:.4f}")
    print("="*40)
    
    # If metrics match user expectations, generate plots
    plot_studio_visuals(yt, yp, phi, threshold=0.5)
else:
    print("\n⚠️ Dataset empty or failed to load. Check H5_DIR and LABELS_CSV.")

print(f"\n✨ All assets saved to: {OUTPUT_DIR}")